In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://books.toscrape.com/" 

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "article.product_pod" 
SELECTOR_DATO_A     = "h3 a"
SELECTOR_DATO_B     = "p.price_color"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) # Tiempo de espera para carga de datos dinámicos

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0 # Indicador de registro procesado (float)
            })
        except:
            # Si un elemento está vacío o es un anuncio, saltar al siguiente
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS (Para análisis posterior)
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head()) # Muestra de los primeros 5 datos
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
 Se encontraron 20 registros potenciales.
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
                variable_nombre variable_valor  status
0            A Light in the ...           5177     1.0
1            Tipping the Velvet           5374     1.0
2                    Soumission           5010     1.0
3                 Sharp Objects           4782     1.0
4  Sapiens: A Brief History ...           5423     1.0
 Navegador cerrado.


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACI N DEL DRIVER (EL CUERPO) ---
# Se define al principio para evitar abrir m ltiples navegadores innecesariamente
options = Options()
options.add_argument("--headless") # Ejecuci n invisible en el servidor Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
# --- PASO 2: INICIALIZACI N DE VARIABLES (LA MEMORIA) ---
# La lista debe estar FUERA del bucle para acumular los datos de todas las p ginas
datos_totales = []
paginas_objetivo = 5 # L m i t e para el ejercicio de laboratorio

try:

    # URL inicial del caso de estudio (E-commerce, Inmobiliario, etc.)
    driver.get("https://www.ejemplo-de-tu-proyecto.com")
   
    # --- PASO 3: BUCLE DE PAGINACI N ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando P g i n a {i + 1} ---")
   
    # Espera expl cita para asegurar que los elementos carguen ( Aprendizaje Continuo)
    WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "article")))
   
   
   
    # CAPTURA: Extraer elementos de la vista actual
    productos = driver.find_elements(By.CLASS_NAME, "article")
    for p in productos:
        datos_totales.append(p.text) # Guardamos en nuestra "mochila"
   
        # NAVEGACI N: L g i c a para saltar a la siguiente p g i n a
        try:
            # Buscamos el b o t n "Siguiente" usando XPATH profesional
            boton_siguiente = driver.find_element(By.XPATH, "//span[contains(text(),’Siguiente’)] | //a[@title=’Siguiente’]")
           
            # Hacemos clic y esperamos la carga de la nueva interfaz
            boton_siguiente.click()
            time.sleep(3) # Pausa estrat gica para el renderizado de
            JavaScript
           
        except Exception:
            print("Se ha llegado a la ltima p g i n a o el b o t n no es visible.")
            break # Detiene el bucle si ya no hay m s p ginas
           
            print(f"\nProceso finalizado con xito .")
            print(f"Total de registros capturados: {len(datos_totales)}")
       
except Exception as e:
    print(f"Error detectado durante la ejecuci n: {e}")
   
finally:
    # --- PASO 4: CIERRE SEGURO (CRUCIAL PARA DOCKER) ---
    # Esto evita que los procesos de Chrome ’zombis’ agoten la RAM del laboratorio
    driver.quit()

Error detectado durante la ejecuci n: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: chrome=146.0.7680.164)
Stacktrace:
#0 0x5d9480dcfa6a <unknown>
#1 0x5d94807deab5 <unknown>
#2 0x5d94807d59bf <unknown>
#3 0x5d94807c66d1 <unknown>
#4 0x5d94807c84cc <unknown>
#5 0x5d94807c6bca <unknown>
#6 0x5d94807c63f0 <unknown>
#7 0x5d94807c60a4 <unknown>
#8 0x5d94807c3deb <unknown>
#9 0x5d94807c4630 <unknown>
#10 0x5d94807e21ed <unknown>
#11 0x5d948087a4bb <unknown>
#12 0x5d94808797b6 <unknown>
#13 0x5d9480824cbf <unknown>
#14 0x5d9480825a81 <unknown>
#15 0x5d9480d95a64 <unknown>
#16 0x5d9480d98951 <unknown>
#17 0x5d9480d8221e <unknown>
#18 0x5d9480d9951e <unknown>
#19 0x5d9480d68be0 <unknown>
#20 0x5d9480dbc9b8 <unknown>
#21 0x5d9480dbcb88 <unknown>
#22 0x5d9480dce4de <unknown>
#23 0x782ef046dac3 <unknown>

